## Section for Google Colab initiating

In [14]:
! pip install torchinfo

In [7]:
! git clone https://github.com/HR-Supaero/MielPops

Cloning into 'MielPops'...
remote: Enumerating objects: 372, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 372 (delta 45), reused 44 (delta 23), pack-reused 289 (from 2)
Receiving objects: 100% (372/372), 85.20 MiB | 41.90 MiB/s, done.
Resolving deltas: 100% (188/188), done.


In [9]:
! cp  -r /kaggle/input/competitions/reconnaissance-dabeilles-francaises/data /kaggle/working/MielPops

In [12]:
%cd /kaggle/working/MielPops/

/kaggle/working/MielPops


In [13]:
! python /kaggle/working/MielPops/data_augmentation.py

Found folders ['Andrena perimelas', 'Andrena aerinifrons', 'Andrena chengtehensis', 'Andrena rufula', 'Andrena prodigiosa', 'Andrena banksi', 'Andrena florivaga', 'Andrena flavipes', 'Andrena bicolor', 'Andrena dorsata', 'Andrena rudbeckiae', 'Andrena afrensis', 'Andrena nigroaenea', 'Andrena aliciae', 'Andrena cineraria', 'Andrena vaga', 'Andrena hesperia', 'Andrena nitida', 'Andrena villipes', 'Andrena carantonica', 'Andrena lineolata', 'Andrena haemorrhoa', 'Andrena fulva', 'Andrena irana', 'Andrena ventralis', 'Andrena vulpecula', 'Andrena clarkella', 'Andrena limbata', 'Andrena wilkella', 'Andrena plana', 'Andrena hattorfiana', 'Andrena perplexa', 'Andrena fulvata', 'Andrena fortipunctata', 'Andrena denticulata', 'Andrena convallaria', 'Andrena orbitalis', 'Andrena angustitarsata', 'Andrena pinguis', 'Andrena mediovittata', 'Andrena crawfordi', 'Andrena melanochroa', 'Andrena mendica', 'Andrena leucophaea', 'Andrena costillensis', 'Andrena discors', 'Andrena barbareae', 'Andrena d

In [6]:
! pwd

/kaggle/working


In [4]:
! cd /kaggle/working/MielPops/data

/bin/bash: line 1: cd: /kaggle/working/MielPops/data: No such file or directory


In [106]:
%cd MielPops/

[Errno 2] No such file or directory: 'MielPops/'
/content/MielPops


In [107]:
!pwd

/content/MielPops


In [108]:
## Imports

In [38]:
import torch
import torch.nn as nn
import torchinfo
from model.mini_cnn import MiniCNN
from result.repondeur import prediction_to_csv
import torchvision
import numpy as np
import csv
from tqdm import tqdm
from model.loader import load_dataset, load_test_loader

In [47]:
import timm
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

In [40]:
from torch.utils.data import DataLoader, random_split
from torchvision import transforms
from utils.CustomDataset import CustomDataset
from utils.CustomSkibidiDataset import CustomSkibidiDataset

from torchvision.transforms import v2

In [112]:
# Load zip from google drive
# from google.colab import drive
# drive.mount('/content/drive')

In [41]:
# Check for GPU availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✓ Using GPU: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✓ Using Apple Silicon GPU (MPS)")
else:
    device = torch.device("cpu")
    print("⚠ Using CPU - training will be slow!")

✓ Using GPU: Tesla T4


In [ ]:
def train_crossEntropyLoss(train_set, model, epochs):
    """
    Entraine un model à partir d'un train_set donné. Retourne les metrics de l'entrainement
    """

    # Parameters
    model.train()

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=0.03,
        momentum=0.9,
        weight_decay=1e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=epochs+1,
        eta_min=0.01
    )
    # scheduler = CosineAnnealingWarmRestarts(optimizer, 
    #                                     T_0 = 5,# Number of iterations for the first restart
    #                                     T_mult = 2, # A factor increases TiTi​ after a restart
    #                                     eta_min = 0.004) # Minimum learning rate

    accuracies = np.zeros(epochs)
    losses = np.zeros(epochs)

    for epoch in range(epochs):
        total_loss = 0.0
        correct = 0
        total = 0
        pbar = tqdm(
            train_set,
            desc=f"Model training | Epoch {epoch+1}/{epochs}",
            leave=True
        )
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            #prediction
            logits = model(images)
            loss = criterion(logits, labels)
            #propagation du gradient
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            # Calcul des métriques
            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            pbar.set_postfix({
                "batch_loss": f"{loss.item():.4f}",
                "accuracy": f"{correct/total*100:.2f}%"
            })
        accuracies[epoch] = correct/total
        losses[epoch] = total_loss
        if epoch!=0 and epoch % 10 == 0:
            torch.save(model.state_dict(), f"model/EFnet_{epoch}_warmcosine.pth")

        scheduler.step()
    return accuracies, losses

In [49]:


transform = v2.Compose([
    #v2.ToImage(),                  # Convert to Tensor (if it's PIL)
    v2.Resize((224, 224)),
    v2.ToDtype(torch.float32, scale=True), # Scales to float16 [0, 1] # TODO, send to float16
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


# 2. Instantiate your Dataset
"""train_dataset = CustomDataset(
    img_dir='data/',
    annotations_file='data/train.csv',
    transform=transform,
    target_transform=None # ???
)"""

skibidi_dataset = CustomSkibidiDataset(
    img_dir='data/treated_train/',
    csv_mapper='data/treated_train.csv',
    transform=transform,
    target_transform=None # ???
)

#train_size = int(0.99 * len(skibidi_dataset))
#test_size = len(skibidi_dataset) - train_size

#train_dataset, validation_dataset = random_split(skibidi_dataset, [train_size, test_size])

Found folders ['Andrena perimelas', 'Andrena aerinifrons', 'Andrena chengtehensis', 'Andrena rufula', 'Andrena prodigiosa', 'Andrena banksi', 'Andrena florivaga', 'Andrena flavipes', 'Andrena bicolor', 'Andrena dorsata', 'Andrena rudbeckiae', 'Andrena afrensis', 'Andrena nigroaenea', 'Andrena aliciae', 'Andrena cineraria', 'Andrena vaga', 'Andrena hesperia', 'Andrena nitida', 'Andrena villipes', 'Andrena carantonica', 'Andrena lineolata', 'Andrena haemorrhoa', 'Andrena fulva', 'Andrena irana', 'Andrena ventralis', 'Andrena vulpecula', 'Andrena clarkella', 'Andrena limbata', 'Andrena wilkella', 'Andrena plana', 'Andrena hattorfiana', 'Andrena perplexa', 'Andrena fulvata', 'Andrena fortipunctata', 'Andrena denticulata', 'Andrena convallaria', 'Andrena orbitalis', 'Andrena angustitarsata', 'Andrena pinguis', 'Andrena mediovittata', 'Andrena crawfordi', 'Andrena melanochroa', 'Andrena mendica', 'Andrena leucophaea', 'Andrena costillensis', 'Andrena discors', 'Andrena barbareae', 'Andrena d

In [116]:
# convnext tiny
# (niels) : j'ai essayé de l'entrainer de 0, c'était trop long.
# -> Essayer de le refaire avec timm, et seulement en fine tuning
"""
# 1. Use Weights instead of a boolean (modern torchvision style)
# weights = ConvNeXt_Tiny_Weights.DEFAULT
# model = convnext_tiny(weights=weights)

# 2. Freeze parameters ONLY if using pretrained weights
# for param in model.parameters():
#     param.requires_grad = False
# Unfreeze the last stage (Stage 3) for better adaptation
# for param in model.features[7].parameters():
#     param.requires_grad = True

# 3. Correctly access the classifier head
# ConvNeXt's head is a Sequential: (0): LayerNorm, (1): Flatten, (2): Linear
in_features = model.classifier[2].in_features
num_classes = 50
model.classifier[2] = torch.nn.Linear(in_features, num_classes)

"""

"\n# 1. Use Weights instead of a boolean (modern torchvision style)\n# weights = ConvNeXt_Tiny_Weights.DEFAULT \n# model = convnext_tiny(weights=weights)\n\n# 2. Freeze parameters ONLY if using pretrained weights\n# for param in model.parameters():\n#     param.requires_grad = False\n# Unfreeze the last stage (Stage 3) for better adaptation\n# for param in model.features[7].parameters():\n#     param.requires_grad = True\n\n# 3. Correctly access the classifier head\n# ConvNeXt's head is a Sequential: (0): LayerNorm, (1): Flatten, (2): Linear\nin_features = model.classifier[2].in_features\nnum_classes = 50\nmodel.classifier[2] = torch.nn.Linear(in_features, num_classes)\n\n"

## Model & training setup

In [ ]:
#set up training
epochs = 32

# convnext atto
# in12k corresponds to the pre-training (image net 12K)
#model = timm.create_model('convnext_nano.in12k', pretrained=True, num_classes=50)
model = timm.create_model('efficientnetv2_s.ra2_in12k', pretrained=True, num_classes=50)


# Freezing the backbone : we keep the transformers block, but change the classifier
# In timm, the classifier is usually stored in 'model.head'
for name, param in model.named_parameters():
    if "head" not in name : #and "stages.3" not in name : bad idea, it breaks the transformer completely
        param.requires_grad = False
    else :
        print(name)




# Send to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train_set = DataLoader(
    dataset=skibidi_dataset,
    batch_size=32,      # Number of images per batch
    shuffle=True,       # Shuffle every epoch to prevent overfitting
    num_workers=4,      # Number of CPU cores for data loading
    pin_memory=True     # Speeds up transfer to GPU
)

validation_test = DataLoader(
    dataset=validation_dataset,
    batch_size=32,      # Number of images per batch
    shuffle=True,       # Shuffle every epoch to prevent overfitting
    num_workers=4,      # Number of CPU cores for data loading
    pin_memory=True     # Speeds up transfer to GPU
)


skibidi_set = DataLoader(
    dataset=skibidi_dataset,
    batch_size=32,      # Number of images per batch
    shuffle=True,       # Shuffle every epoch to prevent overfitting
    num_workers=4,      # Number of CPU cores for data loading
    pin_memory=True     # Speeds up transfer to GPU
)

head.norm.weight
head.norm.bias
head.fc.weight
head.fc.bias


## MAIN TRAINING LOOP

In [ ]:
#training
# training_curves, training_curves = train_crossEntropyLoss(train_set,model, epochs)
training_curves, training_curves = train_crossEntropyLoss(skibidi_set,model, epochs)

Model training | Epoch 1/32: 100%|██████████| 820/820 [01:19<00:00, 10.31it/s, batch_loss=1.2723, accuracy=78.30%]
Model training | Epoch 2/32: 100%|██████████| 820/820 [01:15<00:00, 10.84it/s, batch_loss=0.2813, accuracy=90.62%]
Model training | Epoch 3/32: 100%|██████████| 820/820 [01:15<00:00, 10.83it/s, batch_loss=0.0185, accuracy=94.02%]
Model training | Epoch 4/32: 100%|██████████| 820/820 [01:15<00:00, 10.87it/s, batch_loss=0.0349, accuracy=96.31%]
Model training | Epoch 5/32: 100%|██████████| 820/820 [01:15<00:00, 10.82it/s, batch_loss=0.1638, accuracy=97.80%]
Model training | Epoch 6/32: 100%|██████████| 820/820 [01:15<00:00, 10.84it/s, batch_loss=0.1858, accuracy=94.87%]
Model training | Epoch 7/32: 100%|██████████| 820/820 [01:15<00:00, 10.84it/s, batch_loss=0.1619, accuracy=95.00%]
Model training | Epoch 8/32: 100%|██████████| 820/820 [01:15<00:00, 10.82it/s, batch_loss=0.1046, accuracy=95.90%]
Model training | Epoch 9/32: 100%|██████████| 820/820 [01:15<00:00, 10.85it/s, b

In [36]:
torch.save(model.state_dict(), f"model/Convnext_Nano_{epochs}_warmcosine.pth")

## Eval results

In [120]:
import csv
import pandas as pd
from eval.Analyser import Analyser

In [121]:
def prediction_to_csv(validation_dataset,model) :
    """
    Charge un dataset et retourne le csv des predictions associé.
    """

    model = model.to(device)
    model.eval()

    id = 0

    # enregistre les predictions
    predictions_array = []

    with torch.no_grad() :

        pbar = tqdm(
            validation_dataset,
            desc="Inference"
        )

        for images, labels in pbar :

            images = images.to(device) # a modifier en fonction du test_set
            img_logits = model(images)


            predictions = img_logits.argmax(dim=1)
            predictions = predictions.cpu().numpy()

            for pred, label in zip(predictions, labels):
                id += 1
                predictions_array.append({
                    "id": id,
                    "pred": int(pred),
                    "true": int(label)
                })

        output_path = f"prediction.csv"

        # Sauvegarde CSV
        with open(output_path, "w", newline='', encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames = ["id", "pred", "true"])
            writer.writeheader()
            writer.writerows(predictions_array)
        print(f"Les predictions ont été sauvegardees dans {output_path}.")

In [122]:
prediction_to_csv(validation_test, model)

Inference: 100%|██████████| 164/164 [00:19<00:00,  8.43it/s]

Les predictions ont été sauvegardees dans prediction.csv.


In [123]:
data = pd.read_csv("prediction.csv")

analyser = Analyser(data)
report = analyser.generate_report()
print(report)

{'f1_score_avg': 0.9228684349074107, 'f1_score_per_class': array([1.        , 0.63874346, 1.        , 0.99638989, 0.71111111,
       0.98947368, 0.99363057, 1.        , 0.96969697, 1.        ,
       0.97959184, 0.62015504, 1.        , 0.95532646, 0.99667774,
       1.        , 0.89051095, 0.72189349, 0.99029126, 1.        ,
       0.98601399, 0.7745098 , 0.99275362, 0.66094421, 0.85507246,
       0.99337748, 0.76595745, 0.99481865, 0.98630137, 1.        ,
       0.83333333, 1.        , 0.99082569, 0.98648649, 0.81339713,
       1.        , 1.        , 0.98859316, 0.81889764, 0.87356322,
       0.98154982, 0.6875    , 0.98961938, 0.96028881, 0.99408284,
       0.98013245, 0.85271318]), 'best_f1': np.int64(0)}
